In [0]:


# COMMAND ----------

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType, DateType, IntegerType
from pyspark.sql.window import Window
from datetime import date

spark = SparkSession.builder.getOrCreate()

# ── Catalog / Schema / Table config ──────────────────────────────────────────
CATALOG        = "hackathon_ltm"
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
QUARANTINE_SCHEMA = "quarantine"

BRONZE_TABLE       = f"{CATALOG}.{BRONZE_SCHEMA}.sp_training_feedback"
SILVER_TABLE       = f"{CATALOG}.{SILVER_SCHEMA}.silver_training_feedback"
QUARANTINE_TABLE   = f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_training_feedback"

# ── Business rules ────────────────────────────────────────────────────────────
RATING_MIN         = 1.0
RATING_MAX         = 5.0
VALID_RECOMMEND    = ["yes", "no"]           # after lower-casing
VALID_COURSE_SFXS  = ["_DB", "_MS", "_CR"]   # known domain suffixes
TODAY              = str(date.today())        # e.g. "2026-06-08"

print(f"Pipeline config loaded. Today = {TODAY}")
print(f"  Bronze  : {BRONZE_TABLE}")
print(f"  Silver  : {SILVER_TABLE}")
print(f"  Quarantine: {QUARANTINE_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📥 Step 1 — Ingest Bronze Table

# COMMAND ----------

df_bronze = spark.read.table(BRONZE_TABLE)

print(f"✅ Ingested {df_bronze.count()} rows from {BRONZE_TABLE}")
print(f"   Columns: {df_bronze.columns}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)


# COMMAND ----------

df_stamped = (
    df_bronze
    .withColumn("pipeline_ingest_ts",  F.current_timestamp())
    .withColumn("source_table",        F.lit(BRONZE_TABLE))
    .withColumn("pipeline_run_date",   F.lit(TODAY).cast(DateType()))
)

print("✅ Audit columns added: pipeline_ingest_ts, source_table, pipeline_run_date")

 |

# COMMAND ----------

# ── Pre-cast date for comparison ──────────────────────────────────────────────
df_flagged = df_stamped.withColumn(
    "_parsed_date", F.to_date(F.col("feedback_date"), "yyyy-MM-dd")
)

# ── Individual quarantine flags ───────────────────────────────────────────────

# Q1: feedback_id null/blank
df_flagged = df_flagged.withColumn(
    "_q1_missing_feedback_id",
    F.col("feedback_id").isNull() | (F.trim(F.col("feedback_id")) == "")
)

# Q2: employee_id null/blank/wrong format (must be E followed by digits)
df_flagged = df_flagged.withColumn(
    "_q2_invalid_employee_id",
    F.col("employee_id").isNull()
    | (F.trim(F.col("employee_id")) == "")
    | (~F.col("employee_id").rlike(r"^E\d+$"))
)

# Q3: course_id null/blank
df_flagged = df_flagged.withColumn(
    "_q3_missing_course_id",
    F.col("course_id").isNull() | (F.trim(F.col("course_id")) == "")
)

# Q4: trainer_name null/blank
df_flagged = df_flagged.withColumn(
    "_q4_missing_trainer_name",
    F.col("trainer_name").isNull() | (F.trim(F.col("trainer_name")) == "")
)

# Q5: rating null
df_flagged = df_flagged.withColumn(
    "_q5_null_rating",
    F.col("rating").isNull()
)

# Q6: rating out of range [1, 5]
df_flagged = df_flagged.withColumn(
    "_q6_invalid_rating_range",
    F.col("rating").isNotNull()
    & ((F.col("rating") < RATING_MIN) | (F.col("rating") > RATING_MAX))
)

# Q7: feedback_date null/blank or completely unparseable
df_flagged = df_flagged.withColumn(
    "_q7_null_feedback_date",
    F.col("feedback_date").isNull()
    | (F.trim(F.col("feedback_date")) == "")
    | F.col("_parsed_date").isNull()
)

# Q8: feedback_date in the future (> today)
df_flagged = df_flagged.withColumn(
    "_q8_future_feedback_date",
    F.col("_parsed_date").isNotNull()
    & (F.col("_parsed_date") > F.lit(TODAY).cast(DateType()))
)

# Q9: recommend null/blank or not in {yes, no} after lower
df_flagged = df_flagged.withColumn(
    "_recommend_normalised", F.lower(F.trim(F.col("recommend")))
)
df_flagged = df_flagged.withColumn(
    "_q9_invalid_recommend",
    F.col("recommend").isNull()
    | (F.trim(F.col("recommend")) == "")
    | (~F.col("_recommend_normalised").isin(VALID_RECOMMEND))
)

# Q10: course_id suffix unknown
df_flagged = df_flagged.withColumn(
    "_course_suffix",
    F.regexp_extract(F.col("course_id"), r"(_[A-Z]+)$", 1)
)
df_flagged = df_flagged.withColumn(
    "_q10_unknown_course_suffix",
    ~F.col("_course_suffix").isin(VALID_COURSE_SFXS)
)

# ── Aggregate quarantine reason (pipe-separated list of violated rules) ────────
df_flagged = df_flagged.withColumn(
    "quarantine_reason",
    F.concat_ws(" | ",
        F.when(F.col("_q1_missing_feedback_id"),    F.lit("Q1:missing_feedback_id")),
        F.when(F.col("_q2_invalid_employee_id"),     F.lit("Q2:invalid_employee_id")),
        F.when(F.col("_q3_missing_course_id"),       F.lit("Q3:missing_course_id")),
        F.when(F.col("_q4_missing_trainer_name"),    F.lit("Q4:missing_trainer_name")),
        F.when(F.col("_q5_null_rating"),             F.lit("Q5:null_rating")),
        F.when(F.col("_q6_invalid_rating_range"),    F.lit("Q6:invalid_rating_range")),
        F.when(F.col("_q7_null_feedback_date"),      F.lit("Q7:null_feedback_date")),
        F.when(F.col("_q8_future_feedback_date"),    F.lit("Q8:future_feedback_date")),
        F.when(F.col("_q9_invalid_recommend"),       F.lit("Q9:invalid_recommend")),
        F.when(F.col("_q10_unknown_course_suffix"),  F.lit("Q10:unknown_course_suffix")),
    )
)

# ── Master quarantine flag ─────────────────────────────────────────────────────
df_flagged = df_flagged.withColumn(
    "_is_quarantined",
    (F.col("quarantine_reason") != "") & F.col("quarantine_reason").isNotNull()
)

print("✅ Quarantine flags evaluated for all 10 rules.")
df_flagged.groupBy("_is_quarantined").count().show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 🔎 Quarantine Reason Breakdown

# COMMAND ----------

from pyspark.sql.functions import col

quarantine_rules = [
    ("Q1 – missing_feedback_id",    "_q1_missing_feedback_id"),
    ("Q2 – invalid_employee_id",    "_q2_invalid_employee_id"),
    ("Q3 – missing_course_id",      "_q3_missing_course_id"),
    ("Q4 – missing_trainer_name",   "_q4_missing_trainer_name"),
    ("Q5 – null_rating",            "_q5_null_rating"),
    ("Q6 – invalid_rating_range",   "_q6_invalid_rating_range"),
    ("Q7 – null_feedback_date",     "_q7_null_feedback_date"),
    ("Q8 – future_feedback_date",   "_q8_future_feedback_date"),
    ("Q9 – invalid_recommend",      "_q9_invalid_recommend"),
    ("Q10 – unknown_course_suffix", "_q10_unknown_course_suffix"),
]

print(f"{'Rule':<40} {'Flagged Rows':>12}")
print("-" * 55)
for label, flag_col in quarantine_rules:
    cnt = df_flagged.filter(F.col(flag_col) == True).count()
    print(f"{label:<40} {cnt:>12}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ✂️ Step 4 — Split: Clean vs Quarantine

# COMMAND ----------

# ── Internal / temp columns to drop before writing ────────────────────────────
_internal_cols = [
    "_parsed_date", "_recommend_normalised", "_course_suffix",
    "_q1_missing_feedback_id", "_q2_invalid_employee_id",
    "_q3_missing_course_id",   "_q4_missing_trainer_name",
    "_q5_null_rating",         "_q6_invalid_rating_range",
    "_q7_null_feedback_date",  "_q8_future_feedback_date",
    "_q9_invalid_recommend",   "_q10_unknown_course_suffix",
    "_is_quarantined",
]

df_clean      = df_flagged.filter(~F.col("_is_quarantined"))
df_quarantine = df_flagged.filter( F.col("_is_quarantined"))

# Drop flag columns from clean; keep quarantine_reason in both for reference
df_clean      = df_clean.drop(*_internal_cols).drop("quarantine_reason")
df_quarantine = df_quarantine.drop(*_internal_cols)

print(f"✅ Split complete:")
print(f"   Clean rows      : {df_clean.count()}")
print(f"   Quarantine rows : {df_quarantine.count()}")



# ── 5A: Standardisation ───────────────────────────────────────────────────────

string_cols = ["feedback_id", "employee_id", "course_id",
               "trainer_name", "feedback_text", "recommend"]

df_silver = df_clean

# Trim all string columns
for c in string_cols:
    df_silver = df_silver.withColumn(c, F.trim(F.col(c)))

# Normalise recommend → Yes / No
df_silver = df_silver.withColumn(
    "recommend",
    F.when(F.lower(F.col("recommend")).isin(["yes", "y"]), F.lit("Yes"))
     .when(F.lower(F.col("recommend")).isin(["no",  "n"]), F.lit("No"))
     .otherwise(F.col("recommend"))   # already validated; this is a safety net
)

# Cast feedback_date to DateType
df_silver = df_silver.withColumn(
    "feedback_date", F.to_date(F.col("feedback_date"), "yyyy-MM-dd")
)

# Ensure rating is DoubleType
df_silver = df_silver.withColumn("rating", F.col("rating").cast("double"))

# Upper-case course_id for consistent JOINs
df_silver = df_silver.withColumn("course_id", F.upper(F.col("course_id")))

# Title-case trainer_name
df_silver = df_silver.withColumn(
    "trainer_name",
    F.initcap(F.col("trainer_name"))
)

# Fill null feedback_text
df_silver = df_silver.withColumn(
    "feedback_text",
    F.when(F.col("feedback_text").isNull(), F.lit("[No feedback provided]"))
     .otherwise(F.col("feedback_text"))
)

print("✅ 5A: Standardisation complete.")

# ── 5B: Derived Columns ───────────────────────────────────────────────────────

# course_domain: DB → "Databricks", MS → "Microsoft Azure", CR → "Coursera"
df_silver = df_silver.withColumn(
    "course_domain",
    F.when(F.col("course_id").endswith("_DB"), F.lit("Databricks"))
     .when(F.col("course_id").endswith("_MS"), F.lit("Microsoft Azure"))
     .when(F.col("course_id").endswith("_CR"), F.lit("Coursera"))
     .otherwise(F.lit("Other"))
)

# course_number: extract numeric part from course_id (e.g. C1001_DB → 1001)
df_silver = df_silver.withColumn(
    "course_number",
    F.regexp_extract(F.col("course_id"), r"C(\d+)_", 1).cast("int")
)

# Temporal derived columns
df_silver = df_silver.withColumn("feedback_year",    F.year(F.col("feedback_date")))
df_silver = df_silver.withColumn("feedback_month",   F.month(F.col("feedback_date")))
df_silver = df_silver.withColumn("feedback_quarter", F.quarter(F.col("feedback_date")))
df_silver = df_silver.withColumn(
    "feedback_year_month",
    F.date_format(F.col("feedback_date"), "yyyy-MM")
)

# rating_band: Low / Medium / High / Excellent
df_silver = df_silver.withColumn(
    "rating_band",
    F.when(F.col("rating") < 3.5,  F.lit("Low"))
     .when(F.col("rating") < 4.0,  F.lit("Medium"))
     .when(F.col("rating") <= 4.5, F.lit("High"))
     .otherwise(F.lit("Excellent"))
)

# is_recommended: Boolean for AVG / COUNT aggregations
df_silver = df_silver.withColumn(
    "is_recommended",
    F.when(F.col("recommend") == "Yes", F.lit(True)).otherwise(F.lit(False))
)

# has_text_feedback: did employee write a comment?
df_silver = df_silver.withColumn(
    "has_text_feedback",
    (F.col("feedback_text") != "[No feedback provided]")
    & F.col("feedback_text").isNotNull()
    & (F.trim(F.col("feedback_text")) != "")
)

# feedback_text_length: character count (useful for engagement score)
df_silver = df_silver.withColumn(
    "feedback_text_length",
    F.when(
        F.col("feedback_text") == "[No feedback provided]", F.lit(0)
    ).otherwise(F.length(F.col("feedback_text")))
)

# trainer_key: normalised surrogate key for trainer dimension
df_silver = df_silver.withColumn(
    "trainer_key",
    F.lower(F.regexp_replace(F.col("trainer_name"), r"[\s\.]", "_"))
)

print("✅ 5B: Derived columns added.")
print("\nFinal Silver schema:")
df_silver.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 📊 Silver — Sample & Profile

# COMMAND ----------

df_silver.show(10, truncate=False)

print("\n── Derived column distributions ──")
df_silver.groupBy("course_domain").count().orderBy(F.desc("count")).show()
df_silver.groupBy("rating_band").count().orderBy(F.desc("count")).show()
df_silver.groupBy("feedback_year", "feedback_quarter").count().orderBy("feedback_year", "feedback_quarter").show()
df_silver.groupBy("is_recommended").count().show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 💾 Step 6A — Write Silver Delta Table

# COMMAND ----------

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"✅ Silver table written: {SILVER_TABLE}")
print(f"   Row count: {spark.read.table(SILVER_TABLE).count()}")



# COMMAND ----------

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{QUARANTINE_SCHEMA}")

(
    df_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"✅ Quarantine table written: {QUARANTINE_TABLE}")
print(f"   Row count: {spark.read.table(QUARANTINE_TABLE).count()}")

# Top quarantine reasons
spark.read.table(QUARANTINE_TABLE).groupBy("quarantine_reason").count() \
    .orderBy(F.desc("count")).show(20, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## ✅ Step 7 — Pipeline Summary

# COMMAND ----------

total_bronze  = df_bronze.count()
total_silver  = spark.read.table(SILVER_TABLE).count()
total_quar    = spark.read.table(QUARANTINE_TABLE).count()
pct_clean     = round(total_silver / total_bronze * 100, 1)
pct_quar      = round(total_quar   / total_bronze * 100, 1)

print("=" * 60)
print("  PIPELINE SUMMARY")
print("=" * 60)
print(f"  Bronze rows ingested  : {total_bronze}")
print(f"  Silver rows written   : {total_silver}  ({pct_clean}%)")
print(f"  Quarantine rows       : {total_quar}   ({pct_quar}%)")
print("-" * 60)
print(f"  Silver table          : {SILVER_TABLE}")
print(f"  Quarantine table      : {QUARANTINE_TABLE}")
print("=" * 60)




Pipeline config loaded. Today = 2026-06-08
  Bronze  : hackathon_ltm.bronze.sp_training_feedback
  Silver  : hackathon_ltm.silver.silver_training_feedback
  Quarantine: hackathon_ltm.quarantine.quarantine_training_feedback
✅ Ingested 500 rows from hackathon_ltm.bronze.sp_training_feedback
   Columns: ['feedback_id', 'employee_id', 'course_id', 'trainer_name', 'rating', 'feedback_text', 'feedback_date', 'recommend', '_ingestion_timestamp', '_source_system', '_source_file']
root
 |-- feedback_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- trainer_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- feedback_text: string (nullable = true)
 |-- feedback_date: date (nullable = true)
 |-- recommend: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)

+-----------+-----------+---------